In [208]:
import pandas as pd
import os
import numpy as np
import plotly.graph_objects as go
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

In [209]:

# Cargar los datos desde el archivo Excel
df = pd.read_csv("../data/outputs/kakebo_merged.csv", sep=";")
df


df

,MES,MONTO,año
0,MES ENERO,14057968,2025
1,MES FEBRERO,3310580,2025
2,MES MARZO,3072536,2025
3,MES ABRIL,3455498,2025
4,MES MAYO,3385952,2025
5,MES JUNIO,3167049,2025
6,MES JULIO,3337777,2025
7,MES AGOSTO,4465156,2025
8,MES SEPTIEMBRE,5206932,2025
9,MES OCTUBRE,6291127,2025


In [210]:
df.mean

<bound method DataFrame.mean of                MES     MONTO   año
0        MES ENERO  14057968  2025
1      MES FEBRERO   3310580  2025
2        MES MARZO   3072536  2025
3        MES ABRIL   3455498  2025
4         MES MAYO   3385952  2025
5        MES JUNIO   3167049  2025
6        MES JULIO   3337777  2025
7       MES AGOSTO   4465156  2025
8   MES SEPTIEMBRE   5206932  2025
9      MES OCTUBRE   6291127  2025
10   MES NOVIEMBRE   5725800  2025
11   MES DICIEMBRE   6068471  2025
12       MES ENERO   8807436  2026
13     MES FEBRERO   4331947  2026
14       MES MARZO         0  2026
15       MES ABRIL         0  2026
16        MES MAYO         0  2026
17       MES JUNIO         0  2026
18       MES JULIO         0  2026
19      MES AGOSTO         0  2026
20  MES SEPTIEMBRE         0  2026
21     MES OCTUBRE         0  2026
22   MES NOVIEMBRE         0  2026
23   MES DICIEMBRE         0  2026>

In [211]:
df.describe()

,MONTO,año
count,2.400000e+01,24.000000
mean,3.111843e+06,2025.500000
std,3.506350e+06,0.510754
min,0.000000e+00,2025.000000
25%,0.000000e+00,2025.000000
50%,3.238814e+06,2025.500000
75%,4.650600e+06,2026.000000
max,1.405797e+07,2026.000000


In [212]:
# Preparar datos (enero 2025 a febrero 2026) y ajustar modelo
df_train = df_real.copy()
df_train["No MES"] = np.arange(len(df_train))

X = df_train[["No MES"]]
y = df_train["MONTO"]

modelo = LinearRegression()
modelo.fit(X, y)

# Predicción del siguiente mes
siguiente_mes = df_train["No MES"].max() + 1
pred_siguiente = modelo.predict([[siguiente_mes]])[0]

pred_siguiente

c:\Users\WAGNER FERNÁNDEZ\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning:

X does not have valid feature names, but LinearRegression was fitted with feature names



np.float64(5312426.934065933)

In [213]:
valor_cop = f"${pred_siguiente:,.0f} COP"
valor_cop

'$5,312,427 COP'

In [214]:
df_pred = df_real.copy()
df_pred = pd.concat(
    [
        df_pred,
        pd.DataFrame(
            {
                "MES": [f"PREDICCION MES {siguiente_mes + 1}"],
                "MONTO": [pred_siguiente],
                "año": [df["año"].max()],
            }
        ),
    ],
    ignore_index=True,
)
df_pred

df_pred["MONTO_COP"] = df_pred["MONTO"].apply(
    lambda x: f"${x:,.0f}".replace(",", ".") + " COP"
)
df_pred

,MES,MONTO,año,MONTO_COP
0,MES ENERO,1.405797e+07,2025,$14.057.968 COP
1,MES FEBRERO,3.310580e+06,2025,$3.310.580 COP
2,MES MARZO,3.072536e+06,2025,$3.072.536 COP
3,MES ABRIL,3.455498e+06,2025,$3.455.498 COP
4,MES MAYO,3.385952e+06,2025,$3.385.952 COP
5,MES JUNIO,3.167049e+06,2025,$3.167.049 COP
6,MES JULIO,3.337777e+06,2025,$3.337.777 COP
7,MES AGOSTO,4.465156e+06,2025,$4.465.156 COP
8,MES SEPTIEMBRE,5.206932e+06,2025,$5.206.932 COP
9,MES OCTUBRE,6.291127e+06,2025,$6.291.127 COP


In [215]:
df_pred.to_csv("../data/outputs/kakebo_pred.csv", sep=";", index=False)

In [216]:
# Colores según sea real o predicción
colores = np.where(df_csv["MES"].str.contains("PREDICCION"), "orange", "steelblue")

fig_bar = go.Figure()
fig_bar.add_trace(
    go.Bar(
        x=df_csv["MES"] + " " + df_csv["año"].astype(str),
        y=df_csv["MONTO"],
        name="Monto",
        marker_color=colores,
        text=df_csv["MONTO_COP"],
        textposition="outside",
    )
)
fig_bar.update_layout(
    title={"text": "ANALISIS KAKEBO", "x": 0.5, "xanchor": "center"},
    xaxis_title="Mes y Año",
    yaxis_title="Monto (COP)",
)
fig_bar
